In [ ]:
import torch
# import seisbench
from torchinfo import summary
import os

In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import matplotlib.pyplot as plt
import torch
device = torch.device("npu" if torch.cuda.is_available() else "cpu")

client = Client("GFZ")

t = UTCDateTime("2007/01/02 05:48:50")
stream = client.get_waveforms(network="CX", station="PB01", location="*", channel="HH?", starttime=t-100, endtime=t+100)

fig = plt.figure(figsize=(15, 5))
ax = fig.add_subplot(111)
for i in range(3):
    ax.plot(stream[i].times(), stream[i].data, label=stream[i].stats.channel)
ax.legend()
plt.show()

In [ ]:
def plot_waveform_and_annotations(stream, annotations):
    fig = plt.figure(figsize=(15, 10))
    axs = fig.subplots(2, 1, sharex=True, gridspec_kw={'hspace': 0})

    offset = annotations[0].stats.starttime - stream[0].stats.starttime
    for i in range(3):
        axs[0].plot(stream[i].times(), stream[i].data, label=stream[i].stats.channel)
        if annotations[i].stats.channel[-1] != "N":  # Do not plot noise curve
            axs[1].plot(annotations[i].times() + offset, annotations[i].data, label=annotations[i].stats.channel)

    axs[0].legend()
    axs[1].legend()
    plt.show()

## PhaseNet

In [ ]:
import sys
sys.path.append('/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1/seismoagent/library/ai_module/')
from phase_picking.model import PhaseNet

model = PhaseNet()

model = model.to(device)

print(summary(model, input_size=(1, 3, 3001),device=device))

x = torch.rand(1, 3, 3001)
with torch.no_grad():
    print(model(x))

In [ ]:
pretrained_path = "/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1/seismoagent/library/ai_module/phase_picking/pretrained/v3/phasenet"
pretrained_list = os.listdir(pretrained_path)

pretrained_name_dict = {}
for file in pretrained_list:
    if ".pt" in file:
        pretrained_name = file
        pretrained_name_dict[pretrained_name] = os.path.join(pretrained_path, file)

pretrained_name_dict

In [ ]:
import torch

state = torch.load(pretrained_name_dict["original.pt.v2"],map_location="cpu")

if isinstance(state, dict) and "state_dict" in state:
    state_dict = state["state_dict"]
else:
    state_dict = state

model.load_state_dict(state_dict)


In [ ]:
annotations = model.annotate(stream)
plot_waveform_and_annotations(stream, annotations)

## BasicPhaseAE

In [ ]:
import sys
sys.path.append('/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1')
from seismoagent.library.ai_module.phase_picking.model import BasicPhaseAE

model = BasicPhaseAE()

model = model.to(device)
print(summary(model, input_size=(1, 3, 3001),device=device))

In [ ]:
pretrained_path = "/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1/seismoagent/library/ai_module/phase_picking/pretrained/v3/basicphaseae"
pretrained_list = os.listdir(pretrained_path)

pretrained_name_dict = {}
for file in pretrained_list:
    if ".pt" in file:
        pretrained_name = file
        pretrained_name_dict[pretrained_name] = os.path.join(pretrained_path, file)

pretrained_name_dict

In [ ]:
import torch

state = torch.load(pretrained_name_dict["stead.pt"],map_location="cpu")

if isinstance(state, dict) and "state_dict" in state:
    state_dict = state["state_dict"]
else:
    state_dict = state

model.load_state_dict(state_dict)


In [ ]:
annotations = model.annotate(stream)
plot_waveform_and_annotations(stream, annotations)

## dpppickerp

In [ ]:
import sys
sys.path.append('/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1')
from seismoagent.library.ai_module.phase_picking.model import DPPPicker

model = DPPPicker(mode="P")

model = model.to(device)

print(summary(model, input_size=(1, 1, 3001),device=device))

In [ ]:
pretrained_path = "/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1/seismoagent/library/ai_module/phase_picking/pretrained/v3/dpppickerp"
pretrained_list = os.listdir(pretrained_path)

pretrained_name_dict = {}
for file in pretrained_list:
    if ".pt" in file:
        pretrained_name = file
        pretrained_name_dict[pretrained_name] = os.path.join(pretrained_path, file)

pretrained_name_dict

In [ ]:
import torch

state = torch.load(pretrained_name_dict["stead.pt"],map_location="cpu")

if isinstance(state, dict) and "state_dict" in state:
    state_dict = state["state_dict"]
else:
    state_dict = state

model.load_state_dict(state_dict)


In [ ]:
import numpy as np
data = np.array([tr.data for tr in stream])
data.shape

In [ ]:
used_data = torch.tensor(data[2],dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
picks = model(used_data)

plt.figure()
plt.subplot(2,1,1)
plt.plot(used_data[0,0,:].cpu().detach().numpy())
plt.subplot(2,1,2)
plt.plot(picks[0,:].cpu().detach().numpy())
plt.show()


## EQtransformer

In [ ]:
import sys
sys.path.append('/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1')
from seismoagent.library.ai_module.phase_picking.model import EQTransformer

model = EQTransformer()

model = model.to(device)
print(summary(model, input_size=(1, 3, 6000),device=device))

In [ ]:
pretrained_path = "/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1/seismoagent/library/ai_module/phase_picking/pretrained/v3/eqtransformer"
pretrained_list = os.listdir(pretrained_path)

pretrained_name_dict = {}
for file in pretrained_list:
    if ".pt" in file:
        pretrained_name = file
        pretrained_name_dict[pretrained_name] = os.path.join(pretrained_path, file)

pretrained_name_dict

In [ ]:
import torch

state = torch.load(pretrained_name_dict["stead.pt.v2"],map_location="cpu")

if isinstance(state, dict) and "state_dict" in state:
    state_dict = state["state_dict"]
else:
    state_dict = state

model.load_state_dict(state_dict)


In [ ]:
annotations = model.annotate(stream)
plot_waveform_and_annotations(stream, annotations)

## GPD

In [ ]:
import sys
sys.path.append('/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1')
from seismoagent.library.ai_module.phase_picking.model import GPD

model = GPD()

model = model.to(device)
print(model.in_channels)
print(model.in_samples)
print(summary(model, input_size=(1, 3, 400),device=device))

In [ ]:
x = torch.rand(1, 3, 400)
with torch.no_grad():
    print(model(x))

In [ ]:
pretrained_path = "/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1/seismoagent/library/ai_module/phase_picking/pretrained/v3/gpd"
pretrained_list = os.listdir(pretrained_path)

pretrained_name_dict = {}
for file in pretrained_list:
    if ".pt" in file:
        pretrained_name = file
        pretrained_name_dict[pretrained_name] = os.path.join(pretrained_path, file)

pretrained_name_dict

In [ ]:
import torch

state = torch.load(pretrained_name_dict["stead.pt"],map_location="cpu")

if isinstance(state, dict) and "state_dict" in state:
    state_dict = state["state_dict"]
else:
    state_dict = state

model.load_state_dict(state_dict)


In [ ]:
annotations = model.annotate(stream)
plot_waveform_and_annotations(stream, annotations)

## OBSTransformer

In [ ]:
import sys
sys.path.append('/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1')
from seismoagent.library.ai_module.phase_picking.model import OBSTransformer

model = OBSTransformer()

model = model.to(device)
print(model.in_channels)
print(model.in_samples)
print(summary(model, input_size=(1, 3, 6000),device=device))

In [ ]:
pretrained_path = "/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1/seismoagent/library/ai_module/phase_picking/pretrained/v3/obstransformer"
pretrained_list = os.listdir(pretrained_path)

pretrained_name_dict = {}
for file in pretrained_list:
    if ".pt" in file:
        pretrained_name = file
        pretrained_name_dict[pretrained_name] = os.path.join(pretrained_path, file)

pretrained_name_dict

In [ ]:
import torch

state = torch.load(pretrained_name_dict["obst2024.pt"],map_location="cpu")

if isinstance(state, dict) and "state_dict" in state:
    state_dict = state["state_dict"]
else:
    state_dict = state

model.load_state_dict(state_dict)


In [ ]:
annotations = model.annotate(stream)
plot_waveform_and_annotations(stream, annotations)

## OBSTransformer

In [ ]:
import sys
sys.path.append('/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1')
from seismoagent.library.ai_module.phase_picking.model import PhaseNetLight

model = PhaseNetLight()

model = model.to(device)
print(model.in_channels)
print(model.in_samples)
print(summary(model, input_size=(1, 3, 3001),device=device))

In [ ]:
pretrained_path = "/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1/seismoagent/library/ai_module/phase_picking/pretrained/v3/phasenetlight"
pretrained_list = os.listdir(pretrained_path)

pretrained_name_dict = {}
for file in pretrained_list:
    if ".pt" in file:
        pretrained_name = file
        pretrained_name_dict[pretrained_name] = os.path.join(pretrained_path, file)

pretrained_name_dict

In [ ]:
import torch

state = torch.load(pretrained_name_dict["stead.pt"],map_location="cpu")

if isinstance(state, dict) and "state_dict" in state:
    state_dict = state["state_dict"]
else:
    state_dict = state

model.load_state_dict(state_dict)


In [ ]:
annotations = model.annotate(stream)
plot_waveform_and_annotations(stream, annotations)

## Skynet

In [ ]:
import sys
sys.path.append('/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1')
from seismoagent.library.ai_module.phase_picking.model import Skynet

model = Skynet()

model = model.to(device)
print(model.in_channels)
print(model.in_samples)
print(summary(model, input_size=(1, 3, 30000),device=device))

In [ ]:
pretrained_path = "/liufeng1afs/project/03_LLM/Science_Discovery_Agenet/TRACE-1.1/seismoagent/library/ai_module/phase_picking/pretrained/v3/skynet"
pretrained_list = os.listdir(pretrained_path)

pretrained_name_dict = {}
for file in pretrained_list:
    if ".pt" in file:
        pretrained_name = file
        pretrained_name_dict[pretrained_name] = os.path.join(pretrained_path, file)

pretrained_name_dict

In [ ]:
import torch

state = torch.load(pretrained_name_dict["original.pt.v1"],map_location="cpu")

if isinstance(state, dict) and "state_dict" in state:
    state_dict = state["state_dict"]
else:
    state_dict = state

model.load_state_dict(state_dict)


In [ ]:
annotations = model.annotate(stream)
plot_waveform_and_annotations(stream, annotations)